# Skin Cancer Detection — Evaluation & Experiments
**Group Project | HAM10000 Dataset | Two-Stage Hierarchical Pipeline**

**Author:** Caden 

---

This notebook covers the full Experiments section of the report:
1. Experimental Setup
2. Baseline Results
3. Quantitative Metrics
4. Confusion Matrices
5. Model Comparison
6. Hyperparameter Experiments
7. Grad-CAM Qualitative Evaluation
8. Limitations

> **How to use:** When Isam and Haruna finish their models, replace the dummy arrays in the *'Load Predictions'* cell with their real predictions. Everything else runs automatically.

---
## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import sys

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    recall_score, precision_score, f1_score, accuracy_score,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Import the evaluation module
sys.path.append(os.path.dirname(os.path.abspath('evaluation.py')))
from evaluation import (
    print_classification_report, print_per_class_recall,
    plot_confusion_matrix, compute_roc_auc, plot_roc_curves,
    compute_f1_scores, evaluate_stage1, evaluate_stage2,
    evaluate_pipeline, compare_models, compare_efficiency,
    BINARY_CLASSES, MALIGNANT_CLASSES
)

print('All imports successful.')

---
## 2. Experimental Setup

### Hardware & Environment

Experiments were run on two machines:
- **Gaming PC:** [fill in RAM, GPU specs]
- **Haruna's laptop:** [fill in RAM, GPU specs]

Both had sufficient GPU memory and storage to train and evaluate the models.

### Key Libraries

| Library | Purpose |
|---|---|
| PyTorch | Main deep learning framework |
| Torchvision | MobileNetV2 architecture and image transforms |
| Timm | Vision Transformer (ViT) architecture |
| Scikit-learn | Evaluation metrics (F1, Recall, Confusion Matrix) |
| Matplotlib | Graphs and visualisation |
| pytorch-gradcam | Grad-CAM heatmaps |
| Pandas | HAM10000 label/CSV handling |

### Data Split

The HAM10000 dataset (10,015 images) was split **70/15/15** (stratified):

| Split | % | Images |
|---|---|---|
| Training | 70% | 7,010 |
| Validation | 15% | 1,503 |
| Test | 15% | 1,502 |

Stratified splitting was used to preserve class proportions across all splits — critical given the severe class imbalance in HAM10000 (e.g., nv accounts for ~67% of images).

### Preprocessing & Augmentation

All images were resized to **224×224 pixels** and normalised using ImageNet mean and standard deviation values for compatibility with both pre-trained architectures.

Data augmentation was applied to the training set to address class imbalance and improve generalisation:
- Random horizontal and vertical flips
- Random rotation
- Colour jitter
- Random cropping

This ensures the model learns to recognise lesions under real-world variation and sees sufficient examples of rare malignant classes.

---
## 3. Load Predictions

> **When models are ready:** Replace the dummy arrays below with real predictions from Isam and Haruna.
>
> You need from each model:
> - `y_true_binary` — ground truth: 0=benign, 1=malignant (all test images)
> - `y_pred_binary` — Stage 1 predictions
> - `y_probs_binary` — Stage 1 probabilities, shape (n_test, 2)
> - `y_true_stage2` — ground truth for images Stage 1 flagged malignant (0=akiec, 1=bcc, 2=mel)
> - `y_pred_stage2` — Stage 2 predictions
> - `y_probs_stage2` — Stage 2 probabilities, shape (n_flagged, 3)

In [ ]:
# ----------------------------------------------------------------
# DUMMY DATA — replace with real model predictions when ready
# ----------------------------------------------------------------
np.random.seed(42)
N_TEST = 1502  # HAM10000 15% test split

# Stage 1 ground truth (67% benign, 33% malignant — matches HAM10000)
y_true_binary = np.random.choice([0, 1], size=N_TEST, p=[0.67, 0.33])

# --- MobileNetV2 predictions (dummy) ---
mobilenet_pred_binary = y_true_binary.copy()
flip = np.random.choice(N_TEST, size=int(N_TEST * 0.15), replace=False)
mobilenet_pred_binary[flip] = 1 - mobilenet_pred_binary[flip]
mobilenet_probs_binary = np.random.dirichlet([3, 2], size=N_TEST)

flagged_mn = (mobilenet_pred_binary == 1).sum()
mobilenet_true_stage2 = np.random.choice([0,1,2], size=flagged_mn, p=[0.2,0.3,0.5])
mobilenet_pred_stage2 = mobilenet_true_stage2.copy()
flip2 = np.random.choice(flagged_mn, size=int(flagged_mn*0.2), replace=False)
mobilenet_pred_stage2[flip2] = np.random.choice([0,1,2], size=len(flip2))
mobilenet_probs_stage2 = np.random.dirichlet([2,2,3], size=flagged_mn)

# --- ViT predictions (dummy) ---
vit_pred_binary = y_true_binary.copy()
flip_v = np.random.choice(N_TEST, size=int(N_TEST * 0.12), replace=False)
vit_pred_binary[flip_v] = 1 - vit_pred_binary[flip_v]
vit_probs_binary = np.random.dirichlet([3, 2], size=N_TEST)

flagged_vit = (vit_pred_binary == 1).sum()
vit_true_stage2 = np.random.choice([0,1,2], size=flagged_vit, p=[0.2,0.3,0.5])
vit_pred_stage2 = vit_true_stage2.copy()
flip3 = np.random.choice(flagged_vit, size=int(flagged_vit*0.15), replace=False)
vit_pred_stage2[flip3] = np.random.choice([0,1,2], size=len(flip3))
vit_probs_stage2 = np.random.dirichlet([2,2,3], size=flagged_vit)

print('Predictions loaded.')
print(f'Test set size: {N_TEST} images')
print(f'Malignant cases: {(y_true_binary==1).sum()} ({(y_true_binary==1).mean()*100:.1f}%)')
print(f'Benign cases:    {(y_true_binary==0).sum()} ({(y_true_binary==0).mean()*100:.1f}%)')

---
## 4. Baseline Results

Baseline results show the initial performance of both models **before any hyperparameter tuning**. This gives a reference point to measure how much improvement tuning achieves.

In [ ]:
# Fill in baseline results from Isam and Haruna once available
# These are the numbers before any tuning was applied

baseline_data = {
    'MobileNetV2 (baseline)': {
        'Stage 1 Recall (Mal)': '[fill in]',
        'Stage 1 Macro F1':     '[fill in]',
        'Stage 1 ROC-AUC':      '[fill in]',
        'Stage 2 Macro F1':     '[fill in]',
    },
    'ViT (baseline)': {
        'Stage 1 Recall (Mal)': '[fill in]',
        'Stage 1 Macro F1':     '[fill in]',
        'Stage 1 ROC-AUC':      '[fill in]',
        'Stage 2 Macro F1':     '[fill in]',
    }
}

pd.DataFrame(baseline_data)

---
## 5. Quantitative Metrics

### Why each metric was chosen

| Metric | Importance | Reason |
|---|---|---|
| **Recall (Sensitivity)** | Highest | TP/(TP+FN) — measures how many cancer cases we catch. A low recall means missed cancers, which is clinically dangerous. |
| **F1-Score** | High | 2×(Precision×Recall)/(Precision+Recall) — balances precision and recall. More meaningful than accuracy on imbalanced data as it focuses on the positive (cancer) class. |
| **Precision** | Medium | TP/(TP+FP) — measures how often the model is correct when it predicts cancer. Low precision means too many false alarms and unnecessary follow-ups. |
| **Accuracy** | Low | (TP+TN)/Total — misleading here because HAM10000 is 67% benign. A model predicting 'not cancer' for everything would score 67% accuracy while being useless. |

### 5.1 MobileNetV2 — Stage 1

In [ ]:
mobilenet_s1_results = evaluate_stage1(
    y_true_binary,
    mobilenet_pred_binary,
    mobilenet_probs_binary
)

### 5.2 MobileNetV2 — Stage 2

In [ ]:
mobilenet_s2_results = evaluate_stage2(
    mobilenet_true_stage2,
    mobilenet_pred_stage2,
    mobilenet_probs_stage2
)

### 5.3 ViT — Stage 1

In [ ]:
vit_s1_results = evaluate_stage1(
    y_true_binary,
    vit_pred_binary,
    vit_probs_binary
)

### 5.4 ViT — Stage 2

In [ ]:
vit_s2_results = evaluate_stage2(
    vit_true_stage2,
    vit_pred_stage2,
    vit_probs_stage2
)

---
## 6. Pipeline Evaluation

The pipeline evaluation measures how Stage 1 errors propagate into Stage 2. If Stage 1 misclassifies a malignant lesion as benign, it **never reaches Stage 2** — this is the most dangerous error in the entire system.

### 6.1 MobileNetV2 — Full Pipeline

In [ ]:
mobilenet_pipeline = evaluate_pipeline(
    y_true_binary,
    mobilenet_pred_binary,
    mobilenet_true_stage2,
    mobilenet_pred_stage2
)

### 6.2 ViT — Full Pipeline

In [ ]:
vit_pipeline = evaluate_pipeline(
    y_true_binary,
    vit_pred_binary,
    vit_true_stage2,
    vit_pred_stage2
)

---
## 7. Model Comparison

Side-by-side comparison of MobileNetV2 vs ViT across all metrics.

In [ ]:
# Combine all results for each model
mobilenet_all = {**mobilenet_s1_results, **mobilenet_s2_results, **mobilenet_pipeline}
vit_all       = {**vit_s1_results,       **vit_s2_results,       **vit_pipeline}

# Keep only numeric values for comparison
mobilenet_num = {k: v for k, v in mobilenet_all.items() if isinstance(v, (int, float))}
vit_num       = {k: v for k, v in vit_all.items()       if isinstance(v, (int, float))}

comparison_df = compare_models(mobilenet_num, vit_num)

---
## 8. Hyperparameter Experiments

Results from tuning key hyperparameters. Fill in the values from Isam and Haruna's training runs.

> **Note:** Ask teammates for their training logs showing how these settings affected validation recall and F1.

In [ ]:
# Learning rate experiment results — fill in from training logs
lr_results = pd.DataFrame({
    'Learning Rate': [0.001, 0.0001, 0.00001],
    'MobileNet Val Recall': ['[fill]', '[fill]', '[fill]'],
    'MobileNet Val F1':     ['[fill]', '[fill]', '[fill]'],
    'ViT Val Recall':       ['[fill]', '[fill]', '[fill]'],
    'ViT Val F1':           ['[fill]', '[fill]', '[fill]'],
})
print('Learning Rate Experiments:')
print(lr_results.to_string(index=False))

In [ ]:
# Batch size experiment results
batch_results = pd.DataFrame({
    'Batch Size': [16, 32, 64],
    'MobileNet Val Recall': ['[fill]', '[fill]', '[fill]'],
    'MobileNet Val F1':     ['[fill]', '[fill]', '[fill]'],
    'ViT Val Recall':       ['[fill]', '[fill]', '[fill]'],
    'ViT Val F1':           ['[fill]', '[fill]', '[fill]'],
})
print('Batch Size Experiments:')
print(batch_results.to_string(index=False))

In [ ]:
# Epochs experiment — training curve
# Replace with real training history from Isam/Haruna

epochs = list(range(1, 21))  # adjust to actual number of epochs
mobilenet_train_recall = [0] * 20   # fill with real values
mobilenet_val_recall   = [0] * 20   # fill with real values
vit_train_recall       = [0] * 20   # fill with real values
vit_val_recall         = [0] * 20   # fill with real values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, mobilenet_train_recall, label='Train Recall')
axes[0].plot(epochs, mobilenet_val_recall,   label='Val Recall')
axes[0].set_title('MobileNetV2 — Recall over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Recall')
axes[0].legend()

axes[1].plot(epochs, vit_train_recall, label='Train Recall')
axes[1].plot(epochs, vit_val_recall,   label='Val Recall')
axes[1].set_title('ViT — Recall over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Recall')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 9. Qualitative Evaluation — Grad-CAM Heatmaps

Grad-CAM heatmaps show which regions of the image the model focused on when making its prediction. This provides clinical interpretability — we want the model to focus on the lesion itself, not background artefacts like rulers or ink marks.

**What to look for:**

| Observation | Meaning |
|---|---|
| Heatmap centred on lesion | Model is learning clinically relevant features |
| Heatmap on background/artefacts | Model may not generalise to new cameras/settings |
| Diffuse heatmap across full image | Model is uncertain |
| Focus on lesion border | May indicate ABCD criteria (Asymmetry, Border, Colour, Diameter) |

In [ ]:
# Grad-CAM visualisation
# Ask Isam (MobileNetV2) and Haruna (ViT) for:
#   - A correctly classified malignant image + its heatmap
#   - A misclassified malignant image + its heatmap

# Once you have the heatmap image files, display them like this:
#
# from PIL import Image
#
# fig, axes = plt.subplots(2, 3, figsize=(14, 8))
# fig.suptitle('Grad-CAM Heatmaps: MobileNetV2 vs ViT', fontsize=14)
#
# # Row 1: Correctly classified
# axes[0,0].imshow(Image.open('gradcam/original_correct.jpg'))
# axes[0,0].set_title('Original Image'); axes[0,0].axis('off')
# axes[0,1].imshow(Image.open('gradcam/mobilenet_correct.jpg'))
# axes[0,1].set_title('MobileNetV2 (correct)'); axes[0,1].axis('off')
# axes[0,2].imshow(Image.open('gradcam/vit_correct.jpg'))
# axes[0,2].set_title('ViT (correct)'); axes[0,2].axis('off')
#
# # Row 2: Misclassified
# axes[1,0].imshow(Image.open('gradcam/original_wrong.jpg'))
# axes[1,0].set_title('Original Image'); axes[1,0].axis('off')
# axes[1,1].imshow(Image.open('gradcam/mobilenet_wrong.jpg'))
# axes[1,1].set_title('MobileNetV2 (misclassified)'); axes[1,1].axis('off')
# axes[1,2].imshow(Image.open('gradcam/vit_wrong.jpg'))
# axes[1,2].set_title('ViT (misclassified)'); axes[1,2].axis('off')
#
# plt.tight_layout()
# plt.show()

print('Grad-CAM section ready — uncomment the code above once you have the heatmap images from Isam and Haruna.')

---
## 10. Efficiency Comparison

Comparing computational cost of MobileNetV2 vs ViT — important for the mobile deployment argument.

In [ ]:
# Run this once you have the trained model objects from Isam and Haruna
#
# from evaluation import compare_efficiency
#
# efficiency_df = compare_efficiency(
#     model1=mobilenet_model,          # PyTorch model object
#     model2=vit_model,
#     model1_path='mobilenet.pt',      # saved model file
#     model2_path='vit.pt',
#     input_size=(3, 224, 224),
#     device='cpu',
#     n_runs=100
# )

# Placeholder table until models are available
efficiency_placeholder = pd.DataFrame({
    'MobileNetV2': {
        'Parameters':          '[fill in]',
        'Model Size (MB)':     '[fill in]',
        'FLOPs':               '[fill in]',
        'Inference (ms/img)':  '[fill in]',
    },
    'ViT': {
        'Parameters':          '[fill in]',
        'Model Size (MB)':     '[fill in]',
        'FLOPs':               '[fill in]',
        'Inference (ms/img)':  '[fill in]',
    }
})
print(efficiency_placeholder.to_string())

---
## 11. Limitations

| Limitation | Impact |
|---|---|
| **Single dataset** | Results may not generalise beyond HAM10000. Clinical validation on independent datasets is needed. |
| **Class imbalance** | nv accounts for ~67% of images. Despite augmentation, rare classes (vasc, df) have fewer training examples and the model may struggle with them. |
| **Synthetic augmentation** | Augmentation creates variations of existing images, not genuinely new data. It does not fully solve the imbalance problem. |
| **No clinical trial** | The model has not been tested in a real clinical environment. Real-world factors (lighting, camera quality, hair/artefacts) may degrade performance. |
| **Hardware constraints** | Inference time benchmarks were measured on specific hardware. Performance will vary on lower-spec mobile devices. |
| **Stage 1 leakage** | Any malignant case missed by Stage 1 is never seen by Stage 2. This error cannot be recovered downstream. |

---
## 12. Conclusion (Draft)

> Fill in the bracketed values once models are complete.

This project developed a two-stage hierarchical pipeline for skin cancer detection using the HAM10000 dataset. Stage 1 classifies lesions as benign or malignant, and Stage 2 identifies the malignant subtype (mel, bcc, akiec). Two architectures were compared: MobileNetV2 (optimised for mobile deployment) and Vision Transformer (optimised for global pattern recognition).

**Key findings:**
- MobileNetV2 achieved a Stage 1 recall of **[X]** and a pipeline recall of **[X]**, with an inference time of **[X] ms/image**.
- ViT achieved a Stage 1 recall of **[X]** and a pipeline recall of **[X]**, with an inference time of **[X] ms/image**.
- **[Model name]** is the recommended model for deployment because **[reason — e.g., comparable recall with significantly lower compute cost]**.

**Limitations:** Results are limited to HAM10000 and have not been clinically validated. Class imbalance remains a challenge, particularly for rare lesion types.

**Future work:** Testing on larger, multi-institution datasets; model compression for further mobile optimisation; threshold tuning to push Stage 1 recall above 0.95.